In [2]:
import pandas as pd
df_collision = pd.read_csv(r"C:\Users\lenovo\Downloads\DEPI project\data\processed\dft-road-casualty-statistics-collision-2025-decoded.csv")
df_casualty = pd.read_csv(r"C:\Users\lenovo\Downloads\DEPI project\data\processed\dft-road-casualty-statistics-casualty-2025-decoded.csv")
df_vehicle = pd.read_csv(r"C:\Users\lenovo\Downloads\DEPI project\data\processed\dft-road-casualty-statistics-vehicle-2025-decoded.csv")

C:\Users\lenovo\AppData\Local\Temp\ipykernel_11068\2362679320.py:4: DtypeWarning: Columns (0: propulsion_code, 1: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  df_vehicle = pd.read_csv(r"C:\Users\lenovo\Downloads\DEPI project\data\processed\dft-road-casualty-statistics-vehicle-2025-decoded.csv")


In [3]:
for name, df in [("casualty", df_casualty), ("collision", df_collision), ("vehicle", df_vehicle)]:
    print(f"The number of columns in {name}: {df.shape[1]}")
    print(df.columns)
    print('-' * 50)

The number of columns in casualty: 24
Index(['collision_index', 'collision_year', 'collision_ref_no',
       'vehicle_reference', 'casualty_reference', 'casualty_class',
       'sex_of_casualty', 'age_of_casualty', 'age_band_of_casualty',
       'casualty_severity', 'pedestrian_location', 'pedestrian_movement',
       'car_passenger', 'bus_or_coach_passenger',
       'pedestrian_road_maintenance_worker', 'casualty_type',
       'casualty_imd_decile', 'lsoa_of_casualty', 'enhanced_casualty_severity',
       'casualty_injury_based', 'casualty_adjusted_severity_serious',
       'casualty_adjusted_severity_slight', 'casualty_distance_banding',
       'escooter_flag'],
      dtype='str')
--------------------------------------------------
The number of columns in collision: 44
Index(['collision_index', 'collision_year', 'collision_ref_no',
       'location_easting_osgr', 'location_northing_osgr', 'longitude',
       'latitude', 'police_force', 'collision_severity', 'number_of_vehicles',
    

In [4]:
KEYS = ["collision_index", "vehicle_reference", "casualty_reference"]

UNKNOWN_TOKENS = {
    "-1", "unknown", "not known", "unknown or other",
    "unknown (self reported)", "unknown (self rep only)",
    "unknown vehicle type (self rep only)", "not known or not requested",}

def handle_missing(df):
    df = df.copy()
    for c in df.columns:
        if c in KEYS:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].mask(df[c] == -1)
        else:
            norm = df[c].astype("string").str.strip().str.lower()
            df[c] = df[c].mask(norm.isin(UNKNOWN_TOKENS), "Unknown")
    return df

df_collision = handle_missing(df_collision)
df_casualty = handle_missing(df_casualty)
df_vehicle = handle_missing(df_vehicle)

for name, df in [("collision", df_collision), ("casualty", df_casualty), ("vehicle", df_vehicle)]:
    left_num = [c for c in df.select_dtypes("number").columns if c not in KEYS and (df[c] == -1).any()]
    left_txt = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c]) and (df[c] == "-1").any()]
    print(name, df.shape, "| Remaining -1:", left_num + left_txt)

collision (101525, 44) | Remaining -1: []
casualty (127883, 24) | Remaining -1: []
vehicle (183948, 32) | Remaining -1: []


In [5]:
drop_collision = [
    "collision_year", "collision_ref_no",
    "local_authority_district",                      
    "junction_detail_historic",                     
    "pedestrian_crossing_human_control_historic",
    "pedestrian_crossing_physical_facilities_historic",
    "carriageway_hazards_historic",
    "location_easting_osgr", "location_northing_osgr",   
]

drop_casualty = ["collision_year", "collision_ref_no"]

drop_vehicle = [
    "collision_year", "collision_ref_no",
    "age_of_vehicle",                              
    "vehicle_manoeuvre_historic",
    "vehicle_location_restricted_lane_historic",
    "journey_purpose_of_driver_historic",
]

df_collision.drop(columns=drop_collision, inplace=True)
df_casualty.drop(columns=drop_casualty, inplace=True)
df_vehicle.drop(columns=drop_vehicle, inplace=True)

In [6]:
print("Collision columns:", df_collision.columns.tolist())
print("Casualty columns:", df_casualty.columns.tolist())
print("Vehicle columns:", df_vehicle.columns.tolist())

Collision columns: ['collision_index', 'longitude', 'latitude', 'police_force', 'collision_severity', 'number_of_vehicles', 'number_of_casualties', 'date', 'day_of_week', 'time', 'local_authority_ons_district', 'local_authority_highway', 'local_authority_highway_current', 'first_road_class', 'first_road_number', 'road_type', 'speed_limit', 'junction_detail', 'junction_control', 'second_road_class', 'second_road_number', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'special_conditions_at_site', 'carriageway_hazards', 'urban_or_rural_area', 'did_police_officer_attend_scene_of_accident', 'trunk_road_flag', 'lsoa_of_accident_location', 'enhanced_severity_collision', 'collision_injury_based', 'collision_adjusted_severity_serious', 'collision_adjusted_severity_slight']
Casualty columns: ['collision_index', 'vehicle_reference', 'casualty_reference', 'casualty_class', 'sex_of_casualty', 'age_of_casualty', 'age_band_of_casualty', 'casualty_severity

In [7]:
for c in ["number_of_vehicles", "number_of_casualties", "speed_limit",
          "first_road_number", "second_road_number"]:
    df_collision[c] = df_collision[c].astype("Int64")

df_casualty["age_of_casualty"] = df_casualty["age_of_casualty"].astype("Int64")

for c in ["age_of_driver", "engine_capacity_cc"]:
    df_vehicle[c] = df_vehicle[c].astype("Int64")

In [8]:
import sqlite3

conn = sqlite3.connect("road_safety.db")
conn.execute("PRAGMA foreign_keys = ON")

create_sql = """
DROP TABLE IF EXISTS casualty;
DROP TABLE IF EXISTS vehicle;
DROP TABLE IF EXISTS collision;

CREATE TABLE collision (
    collision_index VARCHAR(13) NOT NULL,
    longitude FLOAT,
    latitude FLOAT,
    police_force VARCHAR(50),
    collision_severity VARCHAR(20),
    number_of_vehicles INT,
    number_of_casualties INT,
    date DATE,
    day_of_week VARCHAR(20),
    time TIME,
    local_authority_ons_district VARCHAR(20),
    local_authority_highway VARCHAR(20),
    local_authority_highway_current VARCHAR(20),
    first_road_class VARCHAR(50),
    first_road_number INT,
    road_type VARCHAR(50),
    speed_limit INT,
    junction_detail VARCHAR(100),
    junction_control VARCHAR(50),
    second_road_class VARCHAR(100),
    second_road_number INT,
    pedestrian_crossing VARCHAR(150),
    light_conditions VARCHAR(50),
    weather_conditions VARCHAR(50),
    road_surface_conditions VARCHAR(50),
    special_conditions_at_site VARCHAR(100),
    carriageway_hazards VARCHAR(150),
    urban_or_rural_area VARCHAR(50),
    did_police_officer_attend_scene_of_accident VARCHAR(150),
    trunk_road_flag VARCHAR(100),
    lsoa_of_accident_location VARCHAR(20),
    enhanced_severity_collision VARCHAR(50),
    collision_injury_based VARCHAR(50),
    collision_adjusted_severity_serious FLOAT,
    collision_adjusted_severity_slight FLOAT,
    PRIMARY KEY (collision_index)
);

CREATE TABLE vehicle (
    collision_index VARCHAR(13) NOT NULL,
    vehicle_reference INT NOT NULL,
    vehicle_type VARCHAR(100),
    towing_and_articulation VARCHAR(50),
    vehicle_manoeuvre VARCHAR(100),
    vehicle_direction_from VARCHAR(20),
    vehicle_direction_to VARCHAR(20),
    vehicle_location_restricted_lane VARCHAR(150),
    junction_location VARCHAR(100),
    skidding_and_overturning VARCHAR(100),
    hit_object_in_carriageway VARCHAR(100),
    vehicle_leaving_carriageway VARCHAR(100),
    hit_object_off_carriageway VARCHAR(50),
    first_point_of_impact VARCHAR(50),
    vehicle_left_hand_drive VARCHAR(20),
    journey_purpose_of_driver VARCHAR(100),
    sex_of_driver VARCHAR(20),
    age_of_driver INT,
    age_band_of_driver VARCHAR(20),
    engine_capacity_cc INT,
    propulsion_code VARCHAR(50),
    generic_make_model VARCHAR(50),
    driver_imd_decile VARCHAR(50),
    lsoa_of_driver VARCHAR(20),
    escooter_flag VARCHAR(50),
    driver_distance_banding VARCHAR(150),
    PRIMARY KEY (collision_index, vehicle_reference),   
    FOREIGN KEY (collision_index) REFERENCES collision (collision_index)   
);

CREATE TABLE casualty (
    collision_index VARCHAR(13) NOT NULL,
    vehicle_reference INT,
    casualty_reference INT NOT NULL,
    casualty_class VARCHAR(50),
    sex_of_casualty VARCHAR(20),
    age_of_casualty INT,
    age_band_of_casualty VARCHAR(20),
    casualty_severity VARCHAR(20),
    pedestrian_location VARCHAR(150),
    pedestrian_movement VARCHAR(150),
    car_passenger VARCHAR(50),
    bus_or_coach_passenger VARCHAR(50),
    pedestrian_road_maintenance_worker VARCHAR(50),
    casualty_type VARCHAR(150),
    casualty_imd_decile VARCHAR(50),
    lsoa_of_casualty VARCHAR(20),
    enhanced_casualty_severity VARCHAR(50),
    casualty_injury_based VARCHAR(50),
    casualty_adjusted_severity_serious FLOAT,
    casualty_adjusted_severity_slight FLOAT,
    casualty_distance_banding VARCHAR(150),
    escooter_flag VARCHAR(100),
    PRIMARY KEY (collision_index, casualty_reference),   
    FOREIGN KEY (collision_index) REFERENCES collision (collision_index),   
    FOREIGN KEY (collision_index, vehicle_reference) REFERENCES vehicle (collision_index, vehicle_reference)  
);
"""
conn.executescript(create_sql)

In [9]:
df_collision.to_sql("collision", conn, if_exists="append", index=False, chunksize=5000)
df_vehicle.to_sql("vehicle", conn, if_exists="append", index=False, chunksize=5000)
df_casualty.to_sql("casualty", conn, if_exists="append", index=False, chunksize=5000)
conn.commit()

In [ ]:
for t in ["collision", "vehicle", "casualty"]:
    print(t, conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])   

print("Foreign key problems:", conn.execute("PRAGMA foreign_key_check").fetchall()) 

collision 101525
vehicle 183948
casualty 127883
Foreign key problems: []


In [11]:
# Query 1: Total collisions and percentage by severity level

collision_severity_dist_query = """
SELECT 
    collision_severity,
    COUNT(*) AS total_collisions,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM collision), 2) AS percentage
FROM collision
GROUP BY collision_severity
ORDER BY total_collisions DESC;
"""

df_collision_severity = pd.read_sql_query(collision_severity_dist_query, conn)
print("=== Collision Severity Distribution ===")
print(df_collision_severity)

=== Collision Severity Distribution ===
  collision_severity  total_collisions  percentage
0             Slight             74881       73.76
1            Serious             25191       24.81
2              Fatal              1453        1.43


In [12]:
# Query 2: Top 5 collision days with average casualties per incident

top_collision_days_query = """
SELECT 
    day_of_week,
    COUNT(*) AS total_collisions,
    ROUND(AVG(number_of_casualties), 2) AS avg_casualties
FROM collision
GROUP BY day_of_week
ORDER BY total_collisions DESC
LIMIT 5;
"""

df_top_collision_days = pd.read_sql_query(top_collision_days_query, conn)
print(df_top_collision_days)

  day_of_week  total_collisions  avg_casualties
0      Friday             16811            1.25
1   Wednesday             15369            1.23
2    Thursday             15304            1.23
3     Tuesday             14880            1.23
4      Monday             14309            1.26


In [13]:
# Query 3: Urban collisions with speed limits over 30 mph grouped by light conditions

urban_high_speed_light_query = """
SELECT 
    light_conditions,
    COUNT(*) AS total_collisions
FROM collision
WHERE urban_or_rural_area = 'Urban'
  AND speed_limit > 30
GROUP BY light_conditions
ORDER BY total_collisions DESC;
"""

df_urban_high_speed_light = pd.read_sql_query(urban_high_speed_light_query, conn)
print(df_urban_high_speed_light)

              light_conditions  total_collisions
0                     Daylight              4315
1        Darkness - lights lit              1646
2       Darkness - no lighting               181
3  Darkness - lighting unknown                83
4      Darkness - lights unlit                60


In [14]:
# Query 4: Most frequent vehicle types involved in fatal and serious collisions

severe_collision_vehicles_query = """
SELECT 
    v.vehicle_type,
    COUNT(DISTINCT c.collision_index) AS severe_collisions_count
FROM collision c
JOIN vehicle v ON c.collision_index = v.collision_index
WHERE c.collision_severity IN ('Fatal', 'Serious')
  AND v.vehicle_type != 'Unknown'
GROUP BY v.vehicle_type
ORDER BY severe_collisions_count DESC;
"""

df_severe_collision_vehicles = pd.read_sql_query(severe_collision_vehicles_query, conn)
print(df_severe_collision_vehicles)

                             vehicle_type  severe_collisions_count
0                                     Car                    20617
1                             Pedal cycle                     4538
2     Van / Goods 3.5 tonnes mgw or under                     2873
3                   Motorcycle over 500cc                     2577
4              Motorcycle 125cc and under                     2205
5           Goods 7.5 tonnes mgw and over                      774
6   Motorcycle over 125cc and up to 500cc                      755
7    Bus or coach (17 or more pass seats)                      748
8                   Taxi/Private hire car                      730
9            Personal powered transporter                      510
10                    Electric motorcycle                      275
11                Motorcycle - unknown cc                      262
12                          Other vehicle                      258
13              Motorcycle 50cc and under                     

In [15]:
# Query 5: Impact of weather conditions on fatal casualties count

fatal_casualties_by_weather_query = """
SELECT 
    c.weather_conditions,
    COUNT(cas.casualty_reference) AS total_fatalities
FROM collision c
JOIN casualty cas ON c.collision_index = cas.collision_index
WHERE cas.casualty_severity = 'Fatal'
  AND c.weather_conditions != 'Unknown'
GROUP BY c.weather_conditions
ORDER BY total_fatalities DESC;
"""

df_fatal_by_weather = pd.read_sql_query(fatal_casualties_by_weather_query, conn)
print(df_fatal_by_weather)

      weather_conditions  total_fatalities
0     Fine no high winds              1319
1  Raining no high winds               148
2                  Other                22
3      Fine + high winds                17
4   Raining + high winds                15
5            Fog or mist                12
6  Snowing no high winds                 2


In [16]:
# Query 6: Top 5 vehicle types involved in incidents and their percentage share

top_vehicle_types_query = """
SELECT 
    vehicle_type,
    COUNT(*) AS total_vehicles,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM vehicle WHERE vehicle_type != 'Unknown'), 2) AS percentage
FROM vehicle
WHERE vehicle_type != 'Unknown'
GROUP BY vehicle_type
ORDER BY total_vehicles DESC
LIMIT 5;
"""

df_top_vehicle_types = pd.read_sql_query(top_vehicle_types_query, conn)
print(df_top_vehicle_types)

                          vehicle_type  total_vehicles  percentage
0                                  Car          123212       67.09
1                          Pedal cycle           17021        9.27
2  Van / Goods 3.5 tonnes mgw or under           12555        6.84
3           Motorcycle 125cc and under            7876        4.29
4                Motorcycle over 500cc            4976        2.71


In [17]:
# Query 7: Driver age summary (average, min, max) grouped by gender

driver_age_by_sex_query = """
SELECT 
    sex_of_driver,
    COUNT(*) AS total_drivers,
    ROUND(AVG(age_of_driver), 1) AS avg_age,
    MIN(age_of_driver) AS min_age,
    MAX(age_of_driver) AS max_age
FROM vehicle
WHERE age_of_driver IS NOT NULL
  AND sex_of_driver != 'Unknown'
GROUP BY sex_of_driver
ORDER BY total_drivers DESC;
"""

df_driver_age_by_sex = pd.read_sql_query(driver_age_by_sex_query, conn)
print(df_driver_age_by_sex)

  sex_of_driver  total_drivers  avg_age  min_age  max_age
0          Male         107208     41.3        1      105
1        Female          45536     43.0        2       98


In [18]:
# Query 8: Top vehicle manoeuvres resulting in skidding or overturning

skidding_manoeuvres_query = """
SELECT 
    vehicle_manoeuvre,
    COUNT(*) AS incident_count
FROM vehicle
WHERE skidding_and_overturning != 'None'
  AND skidding_and_overturning != 'Unknown'
  AND vehicle_manoeuvre != 'Unknown'
GROUP BY vehicle_manoeuvre
ORDER BY incident_count DESC
LIMIT 3;
"""

df_skidding_manoeuvres = pd.read_sql_query(skidding_manoeuvres_query, conn)
print(df_skidding_manoeuvres)

     vehicle_manoeuvre  incident_count
0          Going ahead           92600
1        Turning right           16212
2  Slowing or stopping            9519


In [19]:
# Query 9: Average engine capacity in urban versus rural collision zones

avg_engine_by_area_query = """
SELECT 
    c.urban_or_rural_area,
    COUNT(v.vehicle_reference) AS total_vehicles,
    ROUND(AVG(v.engine_capacity_cc), 0) AS avg_engine_capacity_cc
FROM vehicle v
JOIN collision c ON v.collision_index = c.collision_index
WHERE v.engine_capacity_cc IS NOT NULL
    AND c.urban_or_rural_area != 'Unknown'
GROUP BY c.urban_or_rural_area
ORDER BY avg_engine_capacity_cc DESC;
"""

df_engine_by_area = pd.read_sql_query(avg_engine_by_area_query, conn)
print(df_engine_by_area)

  urban_or_rural_area  total_vehicles  avg_engine_capacity_cc
0               Rural           53374                  1922.0
1         Unallocated               6                  1756.0
2               Urban           85699                  1695.0


In [20]:
# Query 10: Total casualties count generated per vehicle category

casualties_per_vehicle_type_query = """
SELECT 
    v.vehicle_type,
    COUNT(cas.casualty_reference) AS total_casualties
FROM vehicle v
JOIN casualty cas ON v.collision_index = cas.collision_index 
                 AND v.vehicle_reference = cas.vehicle_reference
WHERE v.vehicle_type != 'Unknown'
GROUP BY v.vehicle_type
ORDER BY total_casualties DESC;
"""

df_casualties_by_vehicle_type = pd.read_sql_query(casualties_per_vehicle_type_query, conn)
print(df_casualties_by_vehicle_type)

                             vehicle_type  total_casualties
0                                     Car             80293
1                             Pedal cycle             16760
2              Motorcycle 125cc and under              7651
3     Van / Goods 3.5 tonnes mgw or under              5445
4                   Motorcycle over 500cc              5001
5    Bus or coach (17 or more pass seats)              2766
6                   Taxi/Private hire car              2411
7   Motorcycle over 125cc and up to 500cc              1749
8            Personal powered transporter              1474
9               Motorcycle 50cc and under               789
10          Goods 7.5 tonnes mgw and over               589
11                    Electric motorcycle               584
12                Motorcycle - unknown cc               521
13                          Other vehicle               412
14                       Mobility scooter               339
15        Goods over 3.5t. and under 7.5

In [21]:
# Query 11: Casualties distribution and average age per casualty class

casualty_class_distribution_query = """
SELECT 
    casualty_class,
    COUNT(*) AS total_casualties,
    ROUND(AVG(age_of_casualty), 1) AS avg_age
FROM casualty
WHERE casualty_class != 'Unknown'
GROUP BY casualty_class
ORDER BY total_casualties DESC;
"""

df_casualty_class_dist = pd.read_sql_query(casualty_class_distribution_query, conn)
print(df_casualty_class_dist)

    casualty_class  total_casualties  avg_age
0  Driver or rider             83976     40.2
1        Passenger             23929     33.6
2       Pedestrian             19978     37.6


In [22]:
# Query 12: Count of pedestrian casualties grouped by pedestrian location

pedestrian_casualties_by_location_query = """
SELECT 
    pedestrian_location,
    COUNT(*) AS total_pedestrians
FROM casualty
WHERE casualty_class = 'Pedestrian'
  AND pedestrian_location != 'Unknown'
GROUP BY pedestrian_location
ORDER BY total_pedestrians DESC;
"""

df_pedestrian_locations = pd.read_sql_query(pedestrian_casualties_by_location_query, conn)
print(df_pedestrian_locations)

                                 pedestrian_location  total_pedestrians
0                 In carriageway, crossing elsewhere               7213
1           Crossing on pedestrian crossing facility               3378
2                                On footway or verge               2628
3                       In carriageway, not crossing               2106
4  In centre of carriageway - not on refuge, isla...               1459
5  Crossing elsewhere within 50m. of pedestrian c...               1078
6   On refuge, central island or central reservation                108
7                 Crossing in zig-zag approach lines                 79
8                     Crossing in zig-zag exit lines                 28


In [23]:
# Query 13: Age bands most vulnerable to serious injuries

serious_injuries_by_age_band_query = """
SELECT 
    age_band_of_casualty,
    COUNT(*) AS serious_casualties_count
FROM casualty
WHERE casualty_severity = 'Serious'
  AND age_band_of_casualty != 'Unknown'
GROUP BY age_band_of_casualty
ORDER BY serious_casualties_count DESC;
"""

df_serious_by_age_band = pd.read_sql_query(serious_injuries_by_age_band_query, conn)
print(df_serious_by_age_band)

   age_band_of_casualty  serious_casualties_count
0               26 - 35                      4620
1               36 - 45                      3765
2               16 - 20                      3413
3               56 - 65                      3373
4               46 - 55                      3278
5               21 - 25                      2664
6               66 - 75                      2097
7               Over 75                      2008
8               11 - 15                      1396
9                6 - 10                       595
10                0 - 5                       336


In [25]:
# Query 14: Average casualty age grouped by road surface conditions

avg_casualty_age_by_road_surface_query = """
SELECT 
    c.road_surface_conditions,
    COUNT(cas.casualty_reference) AS total_casualties,
    ROUND(AVG(cas.age_of_casualty), 2) AS avg_casualty_age
FROM casualty cas
JOIN collision c ON cas.collision_index = c.collision_index
WHERE c.road_surface_conditions != 'Unknown'
GROUP BY c.road_surface_conditions
ORDER BY total_casualties DESC;
"""

df_casualty_age_road_surface = pd.read_sql_query(avg_casualty_age_by_road_surface_query, conn)

print(df_casualty_age_road_surface)

  road_surface_conditions  total_casualties  avg_casualty_age
0                     Dry             94608             38.76
1             Wet or damp             29204             38.24
2            Frost or ice              1558             37.41
3                    Snow               203             37.13
4    Flood over 3cm. deep               198             37.03


In [26]:
# Query 15: Casualties count by driver age category (Young drivers vs others)

casualties_by_young_drivers_query = """
SELECT 
    CASE 
        WHEN v.age_of_driver < 25 THEN 'Young Driver (< 25)'
        WHEN v.age_of_driver >= 25 THEN 'Older Driver (>= 25)'
        ELSE 'Unknown Age'
    END AS driver_age_group,
    COUNT(cas.casualty_reference) AS total_casualties
FROM casualty cas
JOIN vehicle v ON cas.collision_index = v.collision_index 
              AND cas.vehicle_reference = v.vehicle_reference
WHERE v.age_of_driver IS NOT NULL
GROUP BY driver_age_group
ORDER BY total_casualties DESC;
"""

df_young_drivers_casualties = pd.read_sql_query(
    casualties_by_young_drivers_query, conn
)
print(df_young_drivers_casualties)

       driver_age_group  total_casualties
0  Older Driver (>= 25)             95535
1   Young Driver (< 25)             24688
